# Mood Group Classifier Training
Notebook version of `train.py` using the updated `Gaming and Mental Health.csv` dataset.

In [ ]:
import json
import pickle
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


In [ ]:
DATA_PATH = Path('data/Gaming and Mental Health.csv')
TARGET = 'mood_group'
OUTPUT_PATH = Path('model.joblib')
OUTPUT_PATH_PKL = Path('model.pkl')
PLOTS_DIR = Path('plots')
TEST_SIZE = 0.2
RANDOM_STATE = 42

NOMINAL_CATEGORICAL_FEATURES = ['gender', 'game_genre', 'gaming_platform']

ORDINAL_OR_BINARY_CATEGORICAL_FEATURES = [
    'withdrawal_symptoms', 'loss_of_other_interests', 'continued_despite_problems',
    'eye_strain', 'back_neck_pain', 'sleep_quality', 'sleep_disruption_frequency',
    'academic_work_performance', 'mood_swing_frequency'
]

ORDINAL_CATEGORY_MAP = {
    'sleep_quality': ['Very Poor', 'Poor', 'Fair', 'Good', 'Insomnia'],
    'sleep_disruption_frequency': ['Never', 'Rarely', 'Sometimes', 'Often', 'Always'],
    'academic_work_performance': ['Failing', 'Poor', 'Below Average', 'Average', 'Good', 'Excellent'],
    'mood_swing_frequency': ['Never', 'Rarely', 'Sometimes', 'Often', 'Daily'],
}

MOOD_GROUP_MAP = {
    'Anxious': 'Anxious', 'Restless': 'Anxious',
    'Depressed': 'Negative', 'Angry': 'Negative', 'Withdrawn': 'Negative', 'Irritable': 'Negative',
    'Normal': 'Neutral',
    'Excited': 'Positive', 'Euphoric': 'Positive',
}


In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    drop_cols = [c for c in ['record_id', 'primary_game', 'gaming_addiction_risk_level'] if c in cleaned.columns]
    if drop_cols:
        cleaned = cleaned.drop(columns=drop_cols)
    return cleaned


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    featured = df.copy()
    featured[TARGET] = featured['mood_state'].map(MOOD_GROUP_MAP)
    unknown = featured.loc[featured[TARGET].isna(), 'mood_state'].dropna().unique().tolist()
    if unknown:
        raise ValueError(f'Unmapped mood_state values found: {unknown}')
    return featured


def build_preprocessor(numeric_features, nominal_features, ordinal_features):
    numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    nominal_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformers = [
        ('num', numeric_transformer, numeric_features),
        ('nominal_cat', nominal_transformer, nominal_features),
    ]

    if ordinal_features:
        ordinal_categories = [ORDINAL_CATEGORY_MAP[c] for c in ordinal_features]
        ordinal_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1)),
        ])
        transformers.append(('ordinal_cat', ordinal_transformer, ordinal_features))

    return ColumnTransformer(transformers=transformers)


In [ ]:
df = pd.read_csv(DATA_PATH)
df = clean_data(df)
df = engineer_features(df)

y = df[TARGET]
X = df.drop(columns=[TARGET, 'mood_state'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

nominal_features = [c for c in NOMINAL_CATEGORICAL_FEATURES if c in X.columns]
ordinal_features = [c for c in ORDINAL_CATEGORY_MAP.keys() if c in X.columns]
boolean_features = [
    c for c in ORDINAL_OR_BINARY_CATEGORICAL_FEATURES
    if c in X.columns and c not in ordinal_features
]

X_train[boolean_features] = X_train[boolean_features].astype(int)
X_test[boolean_features] = X_test[boolean_features].astype(int)

categorical_features = nominal_features + ordinal_features
numerical_features = [c for c in X.columns if c not in (categorical_features + boolean_features)] + boolean_features

print('NOMINAL:', nominal_features)
print('ORDINAL:', ordinal_features)
print('NUMERIC:', numerical_features)
print('train shape:', X_train.shape, 'test shape:', X_test.shape)


In [ ]:
preprocessor = build_preprocessor(numerical_features, nominal_features, ordinal_features)
clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')
pipeline = Pipeline([('preprocessor', preprocessor), ('model', clf)])
pipeline.fit(X_train, y_train)

X_head = pipeline.named_steps['preprocessor'].transform(X_train.head())
if hasattr(X_head, 'toarray'):
    X_head = X_head.toarray()
X_head_df = pd.DataFrame(X_head, columns=pipeline.named_steps['preprocessor'].get_feature_names_out())
X_head_df.head()


In [ ]:
preds = pipeline.predict(X_test)
metrics = {
    'accuracy': accuracy_score(y_test, preds),
    'macro_f1': f1_score(y_test, preds, average='macro'),
    'confusion_matrix': confusion_matrix(y_test, preds).tolist(),
    'classification_report': classification_report(y_test, preds, output_dict=True),
}
print(json.dumps(metrics, indent=2))


In [ ]:
PLOTS_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 6))
y_test.value_counts().sort_values(ascending=True).plot(kind='barh', ax=ax)
ax.set_title('Mood Group Distribution (Test Set)')
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'mood_group_distribution.png', dpi=150)

labels = sorted(y_test.unique().tolist())
cm = confusion_matrix(y_test, preds, labels=labels)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_title('Confusion Matrix')
plt.colorbar(im, ax=ax, label='Count')
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150)

plt.show()


In [ ]:
joblib.dump(pipeline, OUTPUT_PATH)
with OUTPUT_PATH_PKL.open('wb') as f:
    pickle.dump(pipeline, f)
print(f'Saved: {OUTPUT_PATH.resolve()}')
print(f'Saved: {OUTPUT_PATH_PKL.resolve()}')
